<a href="https://colab.research.google.com/github/mannduuu07-png/urban-fire-risk-analysiskorea-fire-frequency-severity-analysis/blob/main/notebooks/05_excess_risk_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 - Excess Risk Analysis

The strongest evidence in this project: for specific regions, does
their high casualty rate survive controlling for what *type* of
place their fires occur in (residential vs industrial vs commercial,
etc.)?

Baseline note: expected casualty rate for each target region is
computed from the national place-type distribution excluding that
region itself, to avoid the region's own extreme values inflating
its own benchmark.

In [1]:
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

df = pd.read_parquet('/content/drive/MyDrive/fire_data/processed/cleaned_fire_data.parquet')

Mounted at /content/drive


In [2]:
def excess_risk_check(data, sido, sigungu):
    """Expected casualty rate = national per-place-type casualty rate
    (excluding the target region) weighted by the target region's own
    place-type mix. Compares this to the region's actual rate."""
    baseline = data[~((data['시도'] == sido) & (data['시군구'] == sigungu))]
    place_risk = baseline.groupby('장소대분류').agg(
        화재건수=('장소대분류', 'count'),
        인명피해합계=('인명피해(명)소계', 'sum'),
    )
    place_risk['casualty_rate'] = place_risk['인명피해합계'] / place_risk['화재건수']

    target = data[(data['시도'] == sido) & (data['시군구'] == sigungu)]
    mix = target['장소대분류'].value_counts(normalize=True)
    expected = (mix * place_risk['casualty_rate']).sum()
    actual = target['인명피해(명)소계'].mean()

    print(f"\n[{sido} {sigungu}]")
    print(f"  Place-type mix (top 3): {dict(mix.head(3).round(3))}")
    print(f"  Expected casualty rate (place-mix only): {expected:.4f}")
    print(f"  Actual casualty rate: {actual:.4f}")
    print(f"  Excess risk ratio: {actual/expected:.2f}x")
    return expected, actual

CHECK_TARGETS = [
    ('충청북도', '청주시상당구'),
    ('인천광역시', '남동구'),
    ('경상북도', '포항시북구'),
]
for sido, sigungu in CHECK_TARGETS:
    excess_risk_check(df, sido, sigungu)


[충청북도 청주시상당구]
  Place-type mix (top 3): {'주거': np.float64(0.327), '생활서비스': np.float64(0.135), '자동차,철도차량': np.float64(0.134)}
  Expected casualty rate (place-mix only): 0.0617
  Actual casualty rate: 0.1396
  Excess risk ratio: 2.26x

[인천광역시 남동구]
  Place-type mix (top 3): {'산업시설': np.float64(0.285), '주거': np.float64(0.234), '생활서비스': np.float64(0.129)}
  Expected casualty rate (place-mix only): 0.0610
  Actual casualty rate: 0.0922
  Excess risk ratio: 1.51x

[경상북도 포항시북구]
  Place-type mix (top 3): {'주거': np.float64(0.246), '기타': np.float64(0.176), '자동차,철도차량': np.float64(0.136)}
  Expected casualty rate (place-mix only): 0.0575
  Actual casualty rate: 0.1044
  Excess risk ratio: 1.82x


## Key findings

- **청주시상당구**: high-severity in both 2015-19 and 2020-24
  sub-periods, and shows a ~2.25x excess risk even after controlling
  for place-type composition. Not explained by a single mass-casualty
  event or by what kind of places its fires occur in.
- **인천 남동구**: emerged as high-severity only in 2020-2024. Its
  fires skew heavily industrial (2x national average), but even after
  accounting for that, actual casualties are ~1.5x the expected rate --
  place composition alone doesn't explain it either.